# P-Delta2 Feature Lab — Faster + More Powerful Ingredients

This Colab performs a controlled **ingredient ablation** on the strongest P-Delta2 direction. It keeps one SmolLM2 attention layer replacement isolated and asks two separate questions:

1. **Power:** which ingredient lowers held-out next-token NLL?
2. **Speed:** which ingredient reduces retrieval work / recurrent overhead without losing the quality margin?

| Ingredient | Mechanism | Intended gain |
|---|---|---|
| **Vectorized curvature preconditioner** | same P-Delta2 recurrence, parallelized inside 32-token chunks | speed without changing the math |
| **Sparse dilated exact retrieval** | exact softmax only at offsets `0,1,2,4,8,16,32` | 7 comparisons/token instead of a dense 32-token local window |
| **Dual-timescale memory** | split the recurrent feature budget into fast and slow memories and mix per query | retain both local syntax and long-range semantics |
| **Lean dual+dilated** | same ideas at feature dim 64 | quality/speed Pareto candidate |
| **torch.compile** | post-training compiler timing of the winner | implementation speed only; never used to select quality |

The benchmark keeps document-disjoint train/validation/test partitions, checks an exact-softmax replacement control, chooses winners on **validation only**, and then opens the test set.

Research motivation: recent efficient-attention work increasingly combines recurrent/linear state, sparse retrieval and chunking rather than relying on one mechanism alone. This implementation is an independent TinyCeNN experiment, not a reproduction of another model.


In [ ]:
import importlib, pathlib, subprocess, sys, tempfile

REF = 'codex/pdelta2-feature-lab-20260914'
WORK = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
REPO = pathlib.Path(tempfile.mkdtemp(prefix='TinyCeNN-feature-lab-', dir=WORK))
subprocess.run(['git','clone','--depth','1','--branch',REF,'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'transformers==4.57.6','datasets>=3,<5','pandas','matplotlib','pytest>=8'], check=True)
for p in (REPO, REPO/'src'):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
importlib.invalidate_caches()
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())


## 1 · Choose the budget

Start with **balanced**. `quick` verifies the pipeline. `strong` is the useful follow-up if an ingredient clearly improves the Pareto frontier.


In [ ]:
from datetime import datetime, timezone
import torch

PROFILE = 'balanced'   # quick | balanced | strong
LAYER = 18
TRAIN_CONTEXT = 256
TEST_CONTEXTS = '256,512,1024'
SEED = 2026

assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = WORK / 'TinyCeNN-feature-results' / f'{PROFILE}-{RUN_ID}'
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUTPUT_DIR)


## 2 · Numerical / causal checks

The vectorized preconditioner must match a slow tokenwise oracle; sparse retrieval must be causal; streaming must agree with full-prefix execution; and checkpoints must reconstruct.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_pdelta2_feature_lab.py')], cwd=REPO, check=True)


## 3 · Run the ingredient screen

Balanced mode compares recurrent-only P-Delta2, dense-window controls, sparse dilated retrieval, dual-timescale memory, and combined lean/quality variants. Quality and efficient winners are locked on validation before test scoring.


In [ ]:
cmd = [
    sys.executable, str(REPO/'scripts/benchmark_pdelta2_feature_lab.py'),
    '--profile', PROFILE, '--layer', str(LAYER),
    '--train-context', str(TRAIN_CONTEXT), '--test-contexts', TEST_CONTEXTS,
    '--seed', str(SEED), '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO, check=True)


## 4 · Results dashboard

- **Lower ΔNLL** = stronger language-model quality.
- **Lower decode/prefill ms** = faster reference implementation.
- **Lower state/KV ratio** = better persistent-memory scaling.
- `memory_efficient_parity` means the paired interval stays inside ±0.02 nats while persistent state is under 50% of Transformer FP16 KV state.


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

val = pd.read_csv(OUTPUT_DIR/'validation_summary.csv')
test = pd.read_csv(OUTPUT_DIR/'test_summary.csv')
report = json.loads((OUTPUT_DIR/'feature_lab_report.json').read_text())
print('Selection:', report['selection'])
display(val.round(6))
display(test.round(6))

base = test[test.context == TRAIN_CONTEXT].copy()
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(val.decode_step_ms, val.validation_delta_nll, s=80)
for _, row in val.iterrows():
    ax.annotate(row['name'], (row.decode_step_ms, row.validation_delta_nll), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.axhline(0, linewidth=1)
ax.axhline(0.02, linewidth=1, linestyle='--')
ax.set_xlabel('Measured decode-step ms (lower is better)')
ax.set_ylabel('Validation ΔNLL vs Transformer')
ax.set_title('P-Delta2 ingredient speed–quality screen')
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(base.state_vs_transformer_fp16, base.delta_nll, s=80)
for _, row in base.iterrows():
    ax.annotate(row['name'], (row.state_vs_transformer_fp16, row.delta_nll), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.axhline(0, linewidth=1); ax.axhline(0.02, linewidth=1, linestyle='--')
ax.set_xlabel('Persistent state / Transformer FP16 KV')
ax.set_ylabel('Held-out ΔNLL')
ax.set_title('Quality–memory Pareto frontier')
plt.show()

print('Winner diagnostics:')
print(json.dumps(report['winner_diagnostics'], indent=2))


## 5 · How to interpret the ingredients

The important comparisons are:

- `pdelta2_dense32_f96` **vs** `pdelta2_dilated32_f96`: does 7-point logarithmic retrieval preserve quality while reducing attention work?
- `pdelta2_none_f96` **vs** `pdelta2_dual_none_f96`: does a fast+slow memory improve recurrence without exact retrieval?
- `pdelta2_dilated32_f96` **vs** `pdelta2_dual_dilated32_f96`: does the two-timescale memory add useful power on top of sparse retrieval?
- `pdelta2_dual_dilated32_f64`: can we shrink the recurrent feature budget and still remain inside the quality margin?
- `preconditioner_speedup` and `compile_speedup`: implementation gains that should not alter the model's quality.

A result is especially interesting if it moves **down and left** on the plots: lower NLL **and** lower runtime/state.


## 6 · Package and download the complete run

The ZIP contains validation/test CSVs, training history, selection, report and all candidate checkpoints.


In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Archive:', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print('Automatic browser download is Colab-only:', exc)
